# Packsize Module — daily production vs standard

**Power Stop LLC · HK1 · Packsize on-demand packaging line**

This notebook drives the same pipeline as `run.py`. Use it to rebuild the dashboard,
inspect the numbers behind any figure, and re-derive the standard.

**Data flow**

```
Carton Production File\   (manual daily drop)  ─┐
                                                 ├─→  packsize_etl  ─→  packsize_data.json
Error Excel File\         (manual daily drop)  ─┘                          │
                                                                            ▼
                                                        Packsize_Dashboard.html
```

Both drop folders are read in full every run and de-duplicated, so dropping a single
day or a complete re-export both work. Nothing is deleted or moved.

**The standard** lives in `config.yaml`. It was derived on 09-Sep-2026 from 11 clean
days using the best-demonstrated method (p75 of clean days), not a historical average.
Full derivation is in `Packsize-Production-Standard.pptx`.


## 1 · Setup

Run once per session.

In [ ]:

import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)

# Folder holding config.yaml and the .py files.
CONFIG_DIR = Path(r"C:\Users\Olatunji.Adeoye\OneDrive - Power Stop, LLC\Documents\HK1 Projects 2026\Packsize Module\Configuration")
sys.path.insert(0, str(CONFIG_DIR))

import pandas as pd, numpy as np
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

from packsize_etl import (_MdLog, load_config, resolve, load_production,
                          load_errors, daily_metrics, rollup, recalibrate,
                          build_payload, run as etl_run)
import build_dashboard

log = _MdLog()
cfg = load_config(CONFIG_DIR / "config.yaml")

print("config :", cfg["_config_path"])
print("root   :", cfg["paths"]["root"])
print("standard:", cfg["standard"]["throughput_cph"], "cartons/labor-hr",
      "(derived", cfg["standard"]["derived_on"] + ")")

## 2 · Pre-flight

Confirms both drop folders exist and have files in them.

In [ ]:

from run import check
check(cfg, log)

## 3 · Ingest

Reads every file in both drop folders. Two things happen here that are easy to miss:

- the **error log is converted from UTC to America/Chicago** — PackNet exports the event
  log in UTC and the production file in local time, so without this every fault lands
  five hours after the cycle it interrupted;
- **shift-boundary EmergencyStop events are separated from genuine faults** — the crew
  uses the e-stop as a power switch, so roughly 45% of e-stop events are just the machine
  being turned on or off.

In [ ]:

prod = load_production(cfg, log)
err  = load_errors(cfg, log)

display(prod.head(3))
display(err.head(3))

### Sanity check the timezone conversion

Events should cluster at shift start (05:30) and stop at shift end (14:00). If they don't, `rules.error_log_is_utc` in config.yaml is set wrong.

In [ ]:

if len(err):
    by_hour = err.groupby(err["ts"].dt.hour).size()
    print("Error events by local hour of day:")
    print(by_hour.to_string())
    peak = by_hour.idxmax()
    print(f"\nPeak hour: {peak}:00 — expect 5 (shift start).",
          "OK" if peak in (5, 6) else "CHECK rules.error_log_is_utc")

## 4 · Daily metrics

One row per production day, scored against the standard.

In [ ]:

daily = daily_metrics(prod, err, cfg, log)

cols = ["date","cartons","cph","availability_pct","utilization_pct",
        "fault_events","trim_pct","optimal_pct","starve_h","met_standard"]
display(daily[cols].tail(15).round(2))

std = cfg["standard"]["throughput_cph"]
print(f"\n{len(daily)} production days")
print(f"met standard ({std} cph): {int(daily['met_standard'].sum())} days "
      f"({daily['met_standard'].mean()*100:.0f}%)")
print(f"mean {daily['cph'].mean():.1f} cph | median {daily['cph'].median():.1f} | best {daily['cph'].max():.1f}")

### Where does the time actually go?

The split that matters: **fault downtime** is a maintenance problem, **starvation** is a flow problem. They have different owners.

In [ ]:

n_mach = len(cfg["shift"]["machines"])
sched  = cfg["shift"]["scheduled_hours_per_machine"] * n_mach

recent = daily[daily["date"] >= daily["date"].max() - pd.Timedelta(days=60)]
run_h, fault_h, starve_h = recent["run_h"].mean(), recent["fault_h"].mean(), recent["starve_h"].mean()
other = max(0, sched - run_h - fault_h - starve_h)

print(f"Average day, last 60 days — {sched:.0f} scheduled machine-hours\n")
for label, v in [("Running", run_h), ("Fault downtime", fault_h),
                 ("Starved / idle", starve_h), ("Short gaps & setup", other)]:
    print(f"  {label:<22} {v:5.2f} h   {v/sched*100:5.1f}%   {'#' * int(v/sched*50)}")

idle = fault_h + starve_h
if idle > 0:
    print(f"\n{starve_h/idle*100:.0f}% of idle time carries no logged fault — the machines are waiting on work.")

## 5 · Period rollups

Rates are recomputed from period totals. Averaging a daily ratio across days of different size gives the wrong answer.

In [ ]:

show = ["label","days","cartons","cartons_per_day","cph","pct_days_met",
        "availability_pct","utilization_pct","faults_per_day","trim_pct"]

for freq, name in [("M","Monthly"), ("Q","Quarterly"), ("Y","Yearly")]:
    print(f"\n=== {name} ===")
    display(rollup(daily, freq)[show].round(1))

## 6 · Build the dashboard

Writes `Packsize_Dashboard.html` in the module root and archives a dated snapshot.

In [ ]:

payload = build_payload(daily, err, cfg)
out = build_dashboard.build(payload, cfg, log)

print("\nOpen:", out)
# import webbrowser; webbrowser.open(out.resolve().as_uri())

## 7 · Re-derive the standard — run monthly

The standard is best-demonstrated: p75 of clean days for throughput, p90 for stretch.
As the clean-day pool grows the percentiles should be refreshed.

This **prints** a YAML block — it does not write. Changing the standard is a decision,
so paste it into `config.yaml` deliberately and note the date.

In [ ]:

new_std = recalibrate(daily, cfg, log)

if new_std:
    cur = cfg["standard"]
    print("\nCurrent vs proposed:")
    for k in ["throughput_cph","throughput_stretch_cph","cartons_per_day",
              "availability_pct","utilization_pct","max_fault_events"]:
        if k in new_std:
            arrow = "->" if new_std[k] != cur.get(k) else "  (no change)"
            print(f"  {k:<24} {cur.get(k)!s:>8}  {arrow}  {new_std[k]}")

## 8 · Ad-hoc questions

Scratch space for whatever comes up on the call. A few that get asked often.

In [ ]:

# --- Which machine is carrying the faults? -------------------------------
if len(err):
    f = err[err["is_fault"]]
    print("Fault events by machine and type:\n")
    display(f.groupby(["message","machine"]).size().unstack(fill_value=0))

In [ ]:

# --- Days that ran past shift end (overtime signal) ----------------------
ot = daily[daily["overrun_h"] > 0.05][["date","cartons","cph","last_ts","overrun_h"]]
if len(ot):
    display(ot.round(2))
    print(f"\n{len(ot)} of {len(daily)} days ran past {cfg['shift']['end']} "
          f"({ot['overrun_h'].sum():.1f} line-hours total).")
else:
    print(f"No day ran past {cfg['shift']['end']} in the loaded window.")

In [ ]:

# --- Box mix and material efficiency by design --------------------------
mix = (prod.groupby("design")
           .agg(cycles=("ts","size"), cartons=("cartons","sum"),
                zfold_sqft=("zfold_area","sum"), trim_sqft=("trim_area","sum"),
                mean_cycle_s=("cycle_sec","mean"))
           .sort_values("cartons", ascending=False))
mix["trim_pct"] = mix["trim_sqft"] / mix["zfold_sqft"] * 100
mix["share_pct"] = mix["cartons"] / mix["cartons"].sum() * 100
display(mix.head(10).round(2))

In [ ]:

# --- z-Fold consumption, for reorder points -----------------------------
days = daily["date"].nunique()
z = (prod.groupby("zfold")["zfold_area"].sum().sort_values(ascending=False) / days)
print(f"Average z-Fold consumed per production day (sq ft), n={days} days:\n")
print(z.round(0).to_string())
print(f"\nTotal: {z.sum():,.0f} sq ft/day  |  {z.sum()*5:,.0f} sq ft/week")

---

### Notes

- **Adding a second shift.** Update `shift.associates` and `shift.paid_hours` in
  `config.yaml`. The labor-hour denominator, and therefore every throughput figure,
  follows automatically. Re-derive the standard afterwards — a two-shift day is not
  comparable to the one-shift days the current standard was built from.

- **If PackNet renames an export column**, edit the mapping under `columns:` in
  `config.yaml`. Nothing in the Python needs to change.

- **Produced-unit-on-hand** is not in this pipeline. PackNet records what was produced,
  never what was consumed, so on-hand needs pack-out or shipment confirmations from the
  WMS. Once that feed exists it becomes a third drop folder and a fourth KPI tile.
